# Banyak tabel → satu tabel fitur (pola umum, domain apa pun)

Model hanya membaca satu tabel rata: **1 baris = 1 sampel yang ditebak**. Tabel lain diringkas
per kunci lalu ditempel ke tabel utama. Kenali dulu jenis tiap tabel dengan `cek_kunci` (sel 2):

| jenis tabel | ciri | cara | sel |
|---|---|---|---|
| utama (`train`/`test`) | 1 baris per sampel, ada target | train+test digabung | 3 |
| 1-ke-1 / statis (profil, wilayah) | maks 1 baris per kunci | `gabung_aman` | 4 |
| panjang (nilai per jenis: vital) | banyak baris + kolom "jenis" | `ke_lebar` | 5 |
| riwayat (transaksi, kunjungan) | banyak baris + tanggal | `sebelum_kejadian` + ringkas | 6–7 |
| kategori/kode bersinonim | satu arti, banyak ejaan | `KAMUS`, `porsi_kategori` | 8 |

Tiga aturan yang paling sering dilanggar:
1. **Cek kunci dulu.** `gabung_aman` memakai `validate="many_to_one"`: error kalau kanan dobel.
2. **Jumlah baris tabel utama tidak boleh berubah** setelah gabung (dicetak tiap gabung).
3. **Waktu.** Dari tabel riwayat hanya baris SEBELUM waktu kejadian/prediksi. Tabel statis yang
   punya tanggal (akun dibuat, kartu terbit) juga dicek: tanggal sesudah kejadian = bocoran.

Contoh pola yang sama: rekam medis (pasien → kunjungan → vital/gejala/riwayat),
e-commerce (pelanggan → pesanan → item), log aplikasi (user → sesi → event),
bank (akun → pinjaman → transaksi).
Data demo di bawah = bank Berka (pinjaman macet). Nama kolomnya hanya contoh; yang dipakai ulang
adalah fungsinya. Baris `# UBAH` = yang diganti untuk soalmu.

## 1. Baca semua CSV di folder (subfolder ikut) → `tabel["nama_file"]`

In [ ]:
import glob
import os

import pandas as pd

DEMO_FOLDER = "data_latihan"   # folder data_latihan (di sebelah repo)
FOLDER = f"{DEMO_FOLDER}/multitabel/berka"            # UBAH: folder data soal
OUTPUT_FOLDER = "../../outputs/tabular/multitabel"   # UBAH

tabel = {}
for path in sorted(glob.glob(f"{FOLDER}/**/*.csv", recursive=True)):
    nama = os.path.splitext(os.path.basename(path))[0]
    tabel[nama] = pd.read_csv(path, low_memory=False)
    print(f"{nama:22s} {str(tabel[nama].shape):14s} {list(tabel[nama].columns)[:7]}")

## 2. `cek_kunci`: tiap kolom kunci di tiap tabel, berapa baris per kunci?
`maks_per_kunci = 1` → tabel 1-ke-1 (gabung langsung). `> 1` → ringkas dulu (sel 5–8).
Kunci yang muncul di train DAN test DAN tabel lain = kunci penghubung.

In [ ]:
AKHIRAN_KUNCI = "id"   # UBAH: kolom yang namanya berakhiran ini dianggap kunci


def cek_kunci(df, kunci):
    per_kunci = df.groupby(kunci).size()
    jenis = "1-ke-1"
    if per_kunci.max() > 1:
        jenis = "banyak (ringkas dulu)"
    return {"kunci": kunci, "baris": len(df), "kunci_unik": len(per_kunci),
            "maks_per_kunci": int(per_kunci.max()), "jenis": jenis}


laporan = []
for nama, df in tabel.items():
    for kolom in df.columns:
        if kolom.lower().endswith(AKHIRAN_KUNCI):
            laporan.append({"tabel": nama} | cek_kunci(df, kolom))
print(pd.DataFrame(laporan).to_string(index=False))

## 3. Tabel utama: train + test digabung (fitur dibuat sekali untuk keduanya)
Kalau waktu kejadian ada di tabel lain (mis. `index_encounters.csv`), tempel dulu dengan
`gabung_aman` (sel 4) lalu jalankan sel ini dari baris `utama[WAKTU]`.

In [ ]:
TRAIN, TEST = "loan_train", "loan_test"   # UBAH: nama tabel di `tabel`
ID = "loan_id"         # UBAH: kunci sampel (1 baris per sampel, kolom submission)
KUNCI = "account_id"   # UBAH: kunci entitas ke tabel lain (akun, pasien, pelanggan)
WAKTU = "date"         # UBAH: waktu kejadian/prediksi
TARGET = "bad"         # UBAH

train = tabel[TRAIN]
test = tabel[TEST]
utama = pd.concat([train.drop(columns=TARGET), test], ignore_index=True)
utama[WAKTU] = pd.to_datetime(utama[WAKTU], utc=True, format="mixed")
print(utama.shape, "| sampel unik:", utama[ID].nunique(), "| entitas unik:", utama[KUNCI].nunique())
print("entitas train yang juga di test:", int(train[KUNCI].isin(test[KUNCI]).sum()),
      "(> 0 → validasi pakai GroupKFold per entitas)")
print(train[TARGET].value_counts(normalize=True).round(3))

## 4. Tabel 1-ke-1: `gabung_aman` (bisa berantai: wilayah → akun → utama)
`awalan` menandai asal kolom (dipakai lagi di sel 10) dan mencegah bentrok nama (`date`).

In [ ]:
def gabung_aman(kiri, kanan, kunci, awalan=""):
    # kanan harus 1 baris per kunci, jumlah baris kiri tidak berubah
    kanan = kanan.set_index(kunci).add_prefix(awalan)
    hasil = kiri.merge(kanan, left_on=kunci, right_index=True, how="left", validate="many_to_one")
    print(f"{awalan or kunci}: +{kanan.shape[1]} kolom | baris {len(kiri)} → {len(hasil)}")
    return hasil


wilayah = tabel["district"][["district_id", "A4", "A11", "A13"]]   # UBAH: kolom yang dipakai
akun = gabung_aman(tabel["account"], wilayah, "district_id", "wil_")
fitur = gabung_aman(utama, akun, KUNCI, "akun_")

# tabel statis bertanggal: tanggal sesudah kejadian = bocoran (harus 0 atau dikosongkan)
akun_tanggal = pd.to_datetime(fitur["akun_date"], utc=True, format="mixed")
fitur["akun_umur_hari"] = (fitur[WAKTU] - akun_tanggal).dt.days
print("akun dibuat SESUDAH kejadian:", int((fitur["akun_umur_hari"] < 0).sum()))

## 5. Tabel panjang → lebar: `ke_lebar` (1 kolom per nilai "jenis")
Contoh lain: vital `(pasien, vital_name, value)` → `vital_heart_rate`, `vital_bmi`, ...
Satuan beda dalam satu jenis (kPa vs mmHg)? Samakan dulu sebelum pivot.

In [ ]:
def ke_lebar(df, kunci, kolom_jenis, kolom_nilai, agg, awalan):
    lebar = df.pivot_table(index=kunci, columns=kolom_jenis, values=kolom_nilai, aggfunc=agg)
    nama = lebar.columns.astype(str).str.strip()
    lebar.columns = awalan + nama.str.lower()
    return lebar.reset_index()


order = tabel["order"].copy()
order["k_symbol"] = order["k_symbol"].fillna("").str.strip()
order["k_symbol"] = order["k_symbol"].replace("", "kosong")
order_lebar = ke_lebar(order, KUNCI, "k_symbol", "amount", "sum", "order_")   # UBAH
fitur = gabung_aman(fitur, order_lebar, KUNCI)
print(list(order_lebar.columns))

## 6. Tabel riwayat: `sebelum_kejadian` (buang baris sesudah waktu kejadian)
Satu entitas bisa punya beberapa sampel (pasien datang 2×): riwayat disalin per sampel lalu
disaring dengan waktu sampel itu, jadi ringkasannya per `ID`, bukan per entitas.

In [ ]:
def sebelum_kejadian(anak, utama, kunci, kolom_waktu):
    kejadian = utama[[ID, kunci, WAKTU]].rename(columns={WAKTU: "waktu_kejadian"})
    anak = anak.copy()
    anak["waktu_anak"] = pd.to_datetime(anak[kolom_waktu], utc=True, format="mixed")
    anak = anak.merge(kejadian, on=kunci, how="inner")
    sebelum = anak["waktu_anak"] < anak["waktu_kejadian"]
    print(f"baris {len(anak)} → sebelum kejadian {int(sebelum.sum())}")
    anak = anak[sebelum].sort_values("waktu_anak")
    anak["hari_sebelum"] = (anak["waktu_kejadian"] - anak["waktu_anak"]).dt.days
    return anak


trans = sebelum_kejadian(tabel["trans"], utama, KUNCI, "date")   # UBAH: tabel, kolom tanggal

## 7. `ringkas_angka`: per sampel → rata-rata, min, maks, std, nilai terakhir, jumlah baris
`hari_sebelum` ikut diringkas: min = jarak ke kejadian terakhir, max = panjang riwayat.
Jendela N hari terakhir = kondisi menjelang kejadian; rasio jendela/seluruh = tren.

In [ ]:
def ringkas_angka(anak, kolom_angka, per, awalan):
    ringkas = anak.groupby(per)[kolom_angka].agg(["mean", "min", "max", "std", "last"])
    ringkas.columns = ringkas.columns.map("_".join)
    ringkas = ringkas.add_prefix(awalan)
    ringkas[awalan + "jumlah"] = anak.groupby(per).size()
    return ringkas.reset_index()


ANGKA = ["amount", "balance", "hari_sebelum"]   # UBAH: kolom angka tabel riwayat
JENDELA_HARI = 90                               # UBAH
fitur = gabung_aman(fitur, ringkas_angka(trans, ANGKA, ID, "trx_"), ID)
baru = trans[trans["hari_sebelum"] <= JENDELA_HARI]
fitur = gabung_aman(fitur, ringkas_angka(baru, ANGKA, ID, "trxbaru_"), ID)
fitur["trx_porsi_baru"] = fitur["trxbaru_jumlah"] / fitur["trx_jumlah"]

## 8. Kategori di tabel riwayat: satukan sinonim/kode dengan `KAMUS`, lalu `porsi_kategori`
Bangun `KAMUS` dari `sorted(anak[kolom].unique())`. Rapikan huruf dulu (`.str.lower()`,
simbol → spasi) supaya "Sore throat" dan "sore-throat" jadi satu. Nilai di luar kamus → `lain`.
Teks bebas per sampel (keluhan, nama obat) → gabung lalu TF-IDF (`teks/klasifikasi/tfidf`).

In [ ]:
def porsi_kategori(anak, kolom, per, awalan):
    porsi = pd.crosstab(anak[per], anak[kolom], normalize="index")
    porsi.columns = awalan + porsi.columns.astype(str)
    return porsi.reset_index()


KAMUS = {"VYBER": "tarik_tunai", "VYBER KARTOU": "tarik_tunai", "VKLAD": "setor",   # UBAH
         "PREVOD Z UCTU": "transfer_masuk", "PREVOD NA UCET": "transfer_keluar"}
trans["kelompok"] = trans["operation"].str.strip().map(KAMUS)
trans["kelompok"] = trans["kelompok"].fillna("lain")
print(trans["kelompok"].value_counts())
fitur = gabung_aman(fitur, porsi_kategori(trans, "kelompok", ID, "porsi_"), ID)
# teks_per_sampel = trans.groupby(ID)["kelompok"].agg(" | ".join)   # untuk TF-IDF

## 9. Matriks fitur
Fitur domain (rasio yang masuk akal di soalmu) ditambah di sini, mis. cicilan / saldo.

In [ ]:
BUANG = [ID, KUNCI, WAKTU, "akun_date", "akun_district_id"]   # UBAH: kunci + tanggal mentah
fitur["waktu_tahun"] = fitur[WAKTU].dt.year
fitur["trx_cicilan_per_saldo"] = fitur["payments"] / fitur["trx_balance_mean"]   # UBAH
X_semua = fitur.drop(columns=BUANG)
for kolom in X_semua.select_dtypes("object").columns:
    X_semua[kolom] = X_semua[kolom].astype("category")
X = X_semua.iloc[:len(train)]
X_test = X_semua.iloc[len(train):]
y = train[TARGET]
print(X.shape, X_test.shape)

## 10. Skor CV per kelompok tabel (pakai awalan dari sel 4–8)
Menunjukkan tabel mana yang menyumbang. Data kecil → 5 fold × 3 seed.

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score

AWALAN = ["akun_", "order_", "trx_", "trxbaru_", "porsi_"]   # UBAH: urutan ditambahkan
SCORING = "roc_auc"   # UBAH: multikelas → "f1_macro" / "roc_auc_ovr"
MODEL = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=7,
                           min_child_samples=10, verbose=-1)
dipakai = []
for kolom in X.columns:
    if not kolom.startswith(tuple(AWALAN)):
        dipakai.append(kolom)
for awalan in [""] + AWALAN:
    for kolom in X.columns:
        if awalan and kolom.startswith(awalan):
            dipakai.append(kolom)
    skor = []
    for seed in [0, 1, 2]:
        folds = StratifiedKFold(5, shuffle=True, random_state=seed)
        skor.append(cross_val_score(MODEL, X[dipakai], y, cv=folds, scoring=SCORING).mean())
    print(f"+{awalan or 'utama':10s} {len(dipakai):3d} fitur | {SCORING} {np.mean(skor):.4f}"
          f" ± {np.std(skor):.4f}")

Lonjakan besar dari tabel TANPA tanggal (di Berka: `order_sipo`) = curigai. Tabel statis sering
potret AKHIR periode data (sesudah kejadian) → bisa berisi akibat dari target. Boleh dipakai bila
tabel yang sama tersedia untuk test, tetapi sebutkan di laporan.

## 11. Latih di semua train → submission

In [ ]:
MODEL.fit(X[dipakai], y)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
submission = pd.DataFrame()
submission[ID] = test[ID]
submission[TARGET] = MODEL.predict_proba(X_test[dipakai])[:, 1]   # UBAH: label → MODEL.predict
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
print(submission.head())

Cek ke kunci jawaban (hanya data demo, hapus saat lomba).

In [ ]:
from sklearn.metrics import roc_auc_score

kunci_jawaban = tabel["loan_test_labels"]
jawaban = submission[ID].map(dict(zip(kunci_jawaban[ID], kunci_jawaban[TARGET])))
print("AUC test:", round(roc_auc_score(jawaban, submission[TARGET]), 4))